# Labs 1 and 2: Naive strategies and UCB

This notebook covers Labs 1 and 2 of the course. It sets up the `seqlearn`
framework, used throughout the course, then implements the strategies of the first two
lectures: greedy, $\varepsilon$-greedy, explore-then-commit (ETC) and UCB. The experiments
check the results of Tutorials 1 and 2 numerically, then compare the strategies: regret
curves, growth in $\log t$, effect of the parameters, behavior when the gap $\Delta$ changes.

Replace the lines `...  # TODO` with your code. Each implementation is followed by a cell of
automatic tests, which must print "all tests passed".

This version runs without installation, for instance on Google Colab (File > Upload
notebook). The cells of the section "Framework" below write the course package `seqlearn`
next to the notebook: run them first, and again after each reconnection, since Colab erases
the files at the end of a session.
Lab 0 (`lab00-colab.ipynb`) covers the numpy, matplotlib and class features used below.

## Framework

The cells of this section write the course package `seqlearn` next to the notebook. Run them, without modifying them, before the rest of the notebook.

In [ ]:
import os

os.makedirs("seqlearn", exist_ok=True)

In [ ]:
%%writefile seqlearn/__init__.py
"""Lab framework for the Sequential Decision course (ESIEE Paris)."""

from .bandits import (
    Agent,
    Bandit,
    BernoulliBandit,
    GaussianBandit,
    UniformAgent,
    pseudo_regret,
    run,
    run_many,
)
from .plotting import plot_mean

In [ ]:
%%writefile seqlearn/bandits.py
"""Stochastic bandits: environments, agent interface, interaction loop."""

import numpy as np


class Bandit:
    """Stochastic K-armed bandit with arm means `means`."""

    def __init__(self, means, rng=None):
        self.means = np.asarray(means, dtype=float)
        self.n_arms = len(self.means)
        self.rng = np.random.default_rng(rng)

    @property
    def best_mean(self):
        return self.means.max()

    @property
    def gaps(self):
        return self.best_mean - self.means

    def pull(self, arm):
        """Pull arm `arm` and return the reward."""
        raise NotImplementedError


class BernoulliBandit(Bandit):
    def pull(self, arm):
        return float(self.rng.random() < self.means[arm])


class GaussianBandit(Bandit):
    def __init__(self, means, sigma=1.0, rng=None):
        super().__init__(means, rng)
        self.sigma = sigma

    def pull(self, arm):
        return self.means[arm] + self.sigma * self.rng.standard_normal()


class Agent:
    """Bandit agent. Subclasses implement `act`.

    `update` keeps track of `t` (number of rounds played), `counts[a]` (number of pulls
    of arm a) and `mu_hat[a]` (empirical mean of arm a, zero until the arm is pulled),
    using the incremental update seen in the lecture.
    """

    def __init__(self, n_arms, rng=None):
        self.n_arms = n_arms
        self.rng = np.random.default_rng(rng)
        self.reset()

    def reset(self):
        self.t = 0
        self.counts = np.zeros(self.n_arms, dtype=int)
        self.mu_hat = np.zeros(self.n_arms)

    def act(self):
        """Return the arm to play at the next round."""
        raise NotImplementedError

    def update(self, arm, reward):
        self.t += 1
        self.counts[arm] += 1
        self.mu_hat[arm] += (reward - self.mu_hat[arm]) / self.counts[arm]


class UniformAgent(Agent):
    def act(self):
        return int(self.rng.integers(self.n_arms))


def run(env, agent, horizon):
    """Let `agent` play on `env` for `horizon` rounds.

    Return two arrays of length `horizon`: arms played and rewards.
    """
    agent.reset()
    arms = np.empty(horizon, dtype=int)
    rewards = np.empty(horizon)
    for t in range(horizon):
        arm = agent.act()
        if not (isinstance(arm, (int, np.integer)) and 0 <= arm < env.n_arms):
            raise ValueError(
                f"act() must return an integer between 0 and {env.n_arms - 1}, "
                f"but returned {arm!r} at round {t + 1}."
            )
        reward = env.pull(arm)
        agent.update(arm, reward)
        arms[t], rewards[t] = arm, reward
    return arms, rewards


def run_many(make_env, make_agent, horizon, n_runs, seed=0):
    """Repeat `run` over `n_runs` independent and reproducible runs.

    `make_env(rng)` builds the environment and `make_agent(n_arms, rng)` the agent. Each
    run gets its own generators, derived from `seed`: two calls with the same seed see the
    same environment randomness, whatever the agent.
    Return two arrays of shape (n_runs, horizon): arms played and rewards.
    """
    arms = np.empty((n_runs, horizon), dtype=int)
    rewards = np.empty((n_runs, horizon))
    for i, child in enumerate(np.random.SeedSequence(seed).spawn(n_runs)):
        env_seed, agent_seed = child.spawn(2)
        env = make_env(np.random.default_rng(env_seed))
        agent = make_agent(env.n_arms, np.random.default_rng(agent_seed))
        arms[i], rewards[i] = run(env, agent, horizon)
    return arms, rewards


def pseudo_regret(arms, gaps):
    """Cumulative pseudo-regret, sum of gaps[A_s] for s <= t, along the last axis."""
    return np.cumsum(np.asarray(gaps)[arms], axis=-1)

In [ ]:
%%writefile seqlearn/plotting.py
"""Shared plotting: course palette and mean curves with uncertainty bands."""

import matplotlib.pyplot as plt
import numpy as np
from cycler import cycler

# Categorical palette, to be used in this order (validated for color vision deficiencies on
# adjacent pairs). Beyond 4 series, prefer several subplots.
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
TEXT = "#52514e"
MUTED = "#a3a29c"
GRID = "#e6e5e1"


def set_style():
    plt.rcParams.update({
        "axes.prop_cycle": cycler(color=COLORS),
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": MUTED,
        "axes.labelcolor": TEXT,
        "axes.titlecolor": TEXT,
        "axes.grid": True,
        "axes.axisbelow": True,
        "grid.color": GRID,
        "grid.linewidth": 0.8,
        "xtick.color": TEXT,
        "ytick.color": TEXT,
        "lines.linewidth": 2,
        "lines.solid_capstyle": "round",
        "legend.frameon": False,
        "legend.labelcolor": TEXT,
        "font.size": 10,
        "figure.dpi": 110,
        "savefig.bbox": "tight",
    })


def plot_mean(ax, curves, label=None, band="se", color=None, end_label=False, label_dy=0, **kwargs):
    """Plot the mean of `curves`, of shape (n_runs, horizon), with an uncertainty band.

    band="se": mean +/- 2 standard errors; band="quantile": 10% and 90% quantiles.
    end_label=True writes `label` at the end of the curve, in addition to the legend,
    shifted vertically by `label_dy` points to avoid overlaps.
    """
    curves = np.asarray(curves, dtype=float)
    t = np.arange(1, curves.shape[1] + 1)
    mean = curves.mean(axis=0)
    if band == "se":
        half = 2 * curves.std(axis=0, ddof=1) / np.sqrt(len(curves))
        low, high = mean - half, mean + half
    elif band == "quantile":
        low, high = np.quantile(curves, [0.1, 0.9], axis=0)
    else:
        raise ValueError("band must be 'se' or 'quantile'.")
    (line,) = ax.plot(t, mean, label=label, color=color, **kwargs)
    ax.fill_between(t, low, high, color=line.get_color(), alpha=0.15, linewidth=0)
    if end_label and label:
        ax.annotate(label, xy=(t[-1], mean[-1]), xytext=(6, label_dy), textcoords="offset points",
                    va="center", color=TEXT, fontsize="small")
    return line

In [ ]:
%%writefile seqlearn/checks.py
"""Automatic tests for the labs. Each function raises AssertionError with an explicit message."""

import numpy as np

from .bandits import Bandit, run


class _FixedBandit(Bandit):
    """Deterministic bandit: arm a always pays means[a]."""

    def pull(self, arm):
        return float(self.means[arm])


class _DecayBandit(Bandit):
    """Arm 0 pays 1 at its first pull and 0 afterwards; arm 1 always pays 0.5."""

    def __init__(self):
        super().__init__([0.0, 0.5])
        self._first = True

    def pull(self, arm):
        if arm == 1:
            return 0.5
        reward, self._first = float(self._first), False
        return reward


def _arms(agent, env, horizon):
    arms, _ = run(env, agent, horizon)
    return arms.tolist()


def _ok(name):
    print(f"{name}: all tests passed.")


def check_pseudo_regret(pseudo_regret):
    gaps = np.array([0.0, 0.1, 0.3])
    arms = np.array([0, 1, 1, 2])
    expected = np.array([0.0, 0.1, 0.2, 0.5])
    out = pseudo_regret(arms, gaps)
    assert out is not None, "pseudo_regret returns nothing."
    assert np.allclose(out, expected), (
        f"For arms={arms.tolist()} and gaps={gaps.tolist()}, "
        f"expected {expected.tolist()}, got {np.asarray(out).tolist()}."
    )
    batch = np.stack([arms, arms[::-1]])
    out = pseudo_regret(batch, gaps)
    assert np.shape(out) == (2, 4), "For arms of shape (n_runs, T), the output must have the same shape."
    assert np.allclose(out, [expected, [0.3, 0.4, 0.5, 0.5]]), (
        "The cumulative sum must run along the last axis (time)."
    )
    _ok("pseudo_regret")


def check_greedy(GreedyAgent):
    got = _arms(GreedyAgent(3), _FixedBandit([0.2, 0.8, 0.5]), 6)
    assert got == [0, 1, 2, 1, 1, 1], (
        f"Noiseless means (0.2, 0.8, 0.5): expected [0, 1, 2, 1, 1, 1], got {got}. "
        "Each arm must be pulled once, in order, before taking the argmax."
    )
    got = _arms(GreedyAgent(2), _FixedBandit([0.5, 0.5]), 4)
    assert got == [0, 1, 0, 0], (
        f"Equal means: expected [0, 1, 0, 0], got {got}. Ties must go to the smallest index."
    )
    agent = GreedyAgent(2)
    _arms(agent, _FixedBandit([0.9, 0.1]), 5)
    got = _arms(agent, _FixedBandit([0.1, 0.9]), 4)
    assert got == [0, 1, 1, 1], "reset() must bring the agent back to its initial state."
    _ok("GreedyAgent")


def check_epsilon_greedy(EpsilonGreedyAgent):
    got = _arms(EpsilonGreedyAgent(3, 0.0), _FixedBandit([0.2, 0.8, 0.5]), 6)
    assert got == [0, 1, 2, 1, 1, 1], f"With epsilon = 0, the agent must behave like greedy: got {got}."
    horizon = 4002
    arms = np.array(_arms(EpsilonGreedyAgent(3, 1.0, rng=0), _FixedBandit([0.2, 0.8, 0.5]), horizon))
    freqs = np.bincount(arms[3:], minlength=3) / (horizon - 3)
    assert np.allclose(freqs, 1 / 3, atol=0.04), (
        f"With epsilon = 1, arms must be uniform: frequencies {freqs.round(3)}."
    )
    arms = np.array(_arms(EpsilonGreedyAgent(2, 0.3, rng=0), _FixedBandit([0.0, 1.0]), horizon))
    freq = np.mean(arms[2:] == 0)
    assert abs(freq - 0.15) < 0.03, (
        "With epsilon = 0.3 and two arms, the bad arm must be played about 15% of the time "
        f"(uniform exploration, hence epsilon / 2): got {100 * freq:.1f}%."
    )
    _ok("EpsilonGreedyAgent")


def check_etc(ETCAgent):
    got = _arms(ETCAgent(3, 2), _FixedBandit([0.3, 0.7, 0.5]), 9)
    assert got == [0, 1, 2, 0, 1, 2, 1, 1, 1], (
        f"ETC(m=2) on (0.3, 0.7, 0.5): expected [0, 1, 2, 0, 1, 2, 1, 1, 1], got {got}. "
        "During exploration, arms are played in turn."
    )
    got = _arms(ETCAgent(2, 3), _FixedBandit([0.5, 0.5]), 8)
    assert got == [0, 1, 0, 1, 0, 1, 0, 0], (
        f"Equal means: expected [0, 1, 0, 1, 0, 1, 0, 0], got {got}. Ties must go to the smallest index."
    )
    got = _arms(ETCAgent(2, 1), _DecayBandit(), 6)
    assert got == [0, 1, 0, 0, 0, 0], (
        f"Expected [0, 1, 0, 0, 0, 0], got {got}. After exploration, ETC plays the same arm "
        "until the end, even if the empirical means change."
    )
    _ok("ETCAgent")


# ---------------------------------------------------------------------------
# Lab 2

def check_ucb(UCBAgent):
    agent = UCBAgent(2, alpha=2.0)
    for arm, reward in [(0, 1.0), (1, 0.0), (0, 0.0)]:
        agent.update(arm, reward)
    out = agent.index()
    assert out is not None, "index returns nothing."
    assert np.shape(out) == (2,), f"index must return an array of length n_arms: got shape {np.shape(out)}."
    expected = np.array([0.5, 0.0]) + np.sqrt(2.0 * np.log(4) / np.array([2, 1]))
    shifted = np.array([0.5, 0.0]) + np.sqrt(2.0 * np.log(3) / np.array([2, 1]))
    assert not np.allclose(out, shifted), (
        "After 3 rounds, the index is the one of round t = 4: the logarithm is log(self.t + 1), "
        "not log(self.t)."
    )
    assert not np.allclose(out, [0.5, 0.0]), "The index is the empirical mean alone: the bonus is missing."
    assert np.allclose(out, expected), (
        f"After the pulls (arm 0, reward 1), (arm 1, reward 0), (arm 0, reward 0), with alpha = 2, "
        f"expected {expected.round(3).tolist()}, got {np.asarray(out).round(3).tolist()}."
    )
    agent = UCBAgent(2, alpha=0.5)
    for arm, reward in [(0, 1.0), (1, 0.0), (0, 0.0)]:
        agent.update(arm, reward)
    expected = np.array([0.5, 0.0]) + np.sqrt(0.5 * np.log(4) / np.array([2, 1]))
    assert np.allclose(agent.index(), expected), (
        f"With alpha = 0.5, expected {expected.round(3).tolist()}, "
        f"got {np.asarray(agent.index()).round(3).tolist()}. The bonus must depend on self.alpha."
    )
    got = _arms(UCBAgent(3, alpha=2.0), _FixedBandit([0.2, 0.8, 0.5]), 3)
    assert got == [0, 1, 2], f"During the first K rounds, each arm must be pulled once, in order: got {got}."
    got = _arms(UCBAgent(3, alpha=2.0), _FixedBandit([0.2, 0.8, 0.5]), 12)
    expected = [0, 1, 2, 1, 2, 1, 0, 1, 2, 1, 1, 2]
    assert got == expected, (
        f"Noiseless means (0.2, 0.8, 0.5), alpha = 2: expected {expected}, got {got}. "
        "After the first K rounds, play the argmax of the index."
    )
    got = _arms(UCBAgent(2, alpha=0.5), _FixedBandit([0.9, 0.1]), 14)
    expected = [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
    assert got == expected, f"Noiseless means (0.9, 0.1), alpha = 0.5: expected {expected}, got {got}."
    agent = UCBAgent(2, alpha=2.0)
    _arms(agent, _FixedBandit([0.9, 0.1]), 5)
    got = _arms(agent, _FixedBandit([0.1, 0.9]), 4)
    assert got == [0, 1, 1, 1], "reset() must bring the agent back to its initial state."
    _ok("UCBAgent")


# ---------------------------------------------------------------------------
# Lab 0

def check_gaps(gaps):
    out = gaps(np.array([0.5, 0.6, 0.4]))
    assert out is not None, "gaps returns nothing."
    assert np.allclose(out, [0.1, 0.0, 0.2]), (
        f"For means (0.5, 0.6, 0.4), expected [0.1, 0.0, 0.2], got {np.asarray(out).tolist()}."
    )
    _ok("gaps")


def check_counts(counts):
    out = counts(np.array([0, 2, 2, 0, 2]), 4)
    assert out is not None, "counts returns nothing."
    assert np.shape(out) == (4,), (
        f"With n_arms = 4, the output must have length 4, even if an arm is never played: got shape {np.shape(out)}."
    )
    assert np.array_equal(out, [2, 0, 3, 0]), (
        f"For arms [0, 2, 2, 0, 2] and n_arms = 4, expected [2, 0, 3, 0], got {np.asarray(out).tolist()}."
    )
    _ok("counts")


def check_running_mean(running_mean):
    x = np.array([1.0, 0.0, 1.0, 1.0])
    expected = np.array([1.0, 0.5, 2 / 3, 0.75])
    out = running_mean(x)
    assert out is not None, "running_mean returns nothing."
    assert np.allclose(out, expected), (
        f"For x = {x.tolist()}, expected {expected.round(3).tolist()}, got {np.asarray(out).round(3).tolist()}."
    )
    batch = np.stack([x, 1 - x])
    out = running_mean(batch)
    assert np.shape(out) == (2, 4), "For x of shape (n_runs, T), the output must have the same shape."
    assert np.allclose(out, [expected, 1 - expected]), "The running mean must be computed along the last axis (time)."
    _ok("running_mean")


def check_bernoulli(bernoulli):
    out = bernoulli(0.3, 20000, np.random.default_rng(0))
    assert out is not None, "bernoulli returns nothing."
    out = np.asarray(out)
    assert out.shape == (20000,), f"With size = 20000, the output must have shape (20000,): got {out.shape}."
    assert np.all((out == 0) | (out == 1)), "The values must be 0 or 1."
    assert abs(out.mean() - 0.3) < 0.02, f"With p = 0.3, the mean of the draws must be close to 0.3: got {out.mean():.3f}."
    again = np.asarray(bernoulli(0.3, 20000, np.random.default_rng(0)))
    assert np.array_equal(out, again), "Two calls with generators of the same seed must return the same draws: use `rng` for all random draws."
    out = np.asarray(bernoulli(0.5, (3, 7), np.random.default_rng(1)))
    assert out.shape == (3, 7), f"With size = (3, 7), the output must have shape (3, 7): got {out.shape}."
    _ok("bernoulli")


def check_running_mean_class(RunningMean):
    m = RunningMean()
    assert m.n == 0, "A new RunningMean must have n = 0."
    values = []
    for x in [1.0, 0.0, 1.0, 1.0]:
        m.update(x)
        values.append(m.value)
    assert m.n == 4, f"After 4 updates, n must be 4: got {m.n}."
    assert np.allclose(values, [1.0, 0.5, 2 / 3, 0.75]), (
        f"After the updates 1, 0, 1, 1, the successive values must be [1, 0.5, 0.667, 0.75]: got {np.round(values, 3).tolist()}."
    )
    stored = [v for v in vars(m).values() if isinstance(v, (list, np.ndarray)) and np.size(v) > 1]
    assert not stored, "The past observations must not be stored: keep only n and value."
    _ok("RunningMean")


def check_round_robin(RoundRobinAgent):
    got = _arms(RoundRobinAgent(3), _FixedBandit([0.2, 0.8, 0.5]), 7)
    assert got == [0, 1, 2, 0, 1, 2, 0], f"With 3 arms, expected [0, 1, 2, 0, 1, 2, 0], got {got}."
    _ok("RoundRobinAgent")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from seqlearn import Agent, BernoulliBandit, UniformAgent, plot_mean, run, run_many
from seqlearn import checks
from seqlearn.plotting import COLORS, MUTED, set_style

set_style()

## 0. Getting started with the framework

An **environment** (`BernoulliBandit`, `GaussianBandit`) has attributes `n_arms`, `means`
and `gaps` (the gaps $\Delta_a$), and a method `pull(arm)` that returns a reward. Arms are
numbered from 0.

An **agent** inherits from `Agent` and implements `act()`, which returns the arm to play at
the next round. The method `update(arm, reward)`, already written, keeps track of `t`
(number of rounds played), `counts` (the $N_a(t)$) and `mu_hat` (the $\hat\mu_a(t)$) with
the incremental update seen in the lecture. Read the code of `Agent` and `UniformAgent` in
`seqlearn/bandits.py`.

`run(env, agent, horizon)` lets the agent play and returns the arms played and the rewards.
`run_many(make_env, make_agent, horizon, n_runs, seed)` repeats the experiment with
independent and reproducible seeds, and returns arrays of shape `(n_runs, horizon)`.

In [ ]:
env = BernoulliBandit([0.6, 0.4], rng=0)
agent = UniformAgent(env.n_arms, rng=1)
arms, rewards = run(env, agent, horizon=10)
print("arms played:", arms)
print("rewards    :", rewards)
print("N_a(10)    :", agent.counts)
print("mu_hat(10) :", agent.mu_hat)
print("gaps       :", env.gaps)

From now on, we work on the instance with two Bernoulli arms of means $0.6$ and $0.4$
(gap $\Delta = 0.2$), with the horizon $T = 5000$ of Tutorial 1.

In [ ]:
MEANS = [0.6, 0.4]
HORIZON = 5000
N_RUNS = 50


def make_env(rng):
    return BernoulliBandit(MEANS, rng=rng)


GAPS = make_env(None).gaps
DELTA = GAPS.max()
t = np.arange(1, HORIZON + 1)

## 1. Pseudo-regret

The pseudo-regret of a run is $\bar R_t = \sum_{s=1}^t \Delta_{A_s}$. By exercise 1 of
Tutorial 1, its expectation is the regret $R_t$.

**Question 1.** Write `pseudo_regret(arms, gaps)`, which returns the cumulative
pseudo-regret for an array `arms` of shape `(T,)` or `(n_runs, T)`.

*Hint:* `gaps[arms]` and `np.cumsum(..., axis=-1)`.

In [ ]:
def pseudo_regret(arms, gaps):
    ...  # TODO


checks.check_pseudo_regret(pseudo_regret)

**Question 2.** Run `UniformAgent` for `N_RUNS` runs. On a single figure, plot the
pseudo-regret of 5 individual runs, its mean with `plot_mean`, and the line
$t \mapsto \frac{t}{K}\sum_a \Delta_a$ obtained in Tutorial 1 (exercise 1, question 3).

In [ ]:
...  # TODO

## 2. Greedy

**Question 3.** Complete `GreedyAgent`: during the first $K$ rounds, play arms
$0, 1, \ldots, K-1$ in order; afterwards, play $\arg\max_a \hat\mu_a$, ties going to the
smallest index (this is what `np.argmax` does). The method must return a Python integer,
for instance `int(np.argmax(...))`.

In [ ]:
class GreedyAgent(Agent):
    def act(self):
        ...  # TODO


checks.check_greedy(GreedyAgent)

**Question 4.** By Tutorial 1 (exercise 2, question 4), the best arm (here arm 0) is
pulled only once if and only if the first two rewards are $0$ then $1$, which happens with
probability $(1-p)q = 0.16$.

1. Over 2000 runs of horizon 50, estimate the fraction of runs in which arm 0 is pulled
   exactly once, with a $\pm 2$ standard error interval, and check that $0.16$ lies in the
   interval.
2. Over 200 runs of horizon `HORIZON`, plot the mean pseudo-regret of greedy and the lower
   bound $(1-p)\,q\,(p-q)\,(t-1)$ of Tutorial 1 (exercise 2, question 3).

In [ ]:
...  # TODO

## 3. $\varepsilon$-greedy

**Question 5.** Complete `EpsilonGreedyAgent`: during the first $K$ rounds, play each arm
once in order; afterwards, with probability `epsilon`, play a uniform arm, and otherwise
the arm with the best empirical mean. Use `self.rng` for all random draws, so that
experiments are reproducible.

In [ ]:
class EpsilonGreedyAgent(Agent):
    def __init__(self, n_arms, epsilon, rng=None):
        self.epsilon = epsilon
        super().__init__(n_arms, rng)

    def act(self):
        ...  # TODO


checks.check_epsilon_greedy(EpsilonGreedyAgent)

## 4. Explore-then-commit

**Question 6.** Complete `ETCAgent`: during the first $mK$ rounds, play the arms in turn
($0, 1, \ldots, K-1, 0, 1, \ldots$); at the next round, choose the arm with the best
empirical mean and play it until the end.

In [ ]:
class ETCAgent(Agent):
    def __init__(self, n_arms, m, rng=None):
        self.m = m
        super().__init__(n_arms, rng)

    def reset(self):
        super().reset()
        self.committed = None

    def act(self):
        ...  # TODO


checks.check_etc(ETCAgent)

The experiments below all have the same form: run an agent for `N_RUNS` runs and compute
its pseudo-regret. The function `regret_curves` does it, for the reference instance by
default, and returns an array of shape `(n_runs, horizon)`. The function `final_stats`
returns the mean final regret and the half-width of its $\pm 2$ standard error interval.

In [ ]:
def regret_curves(make_agent, means=MEANS, horizon=HORIZON, n_runs=N_RUNS):
    arms, _ = run_many(lambda rng: BernoulliBandit(means, rng=rng), make_agent, horizon, n_runs)
    return pseudo_regret(arms, BernoulliBandit(means).gaps)


def final_stats(regret):
    final = regret[:, -1]
    return final.mean(), 2 * final.std(ddof=1) / np.sqrt(len(final))

## 5. Comparison of the naive strategies

**Question 7.** Compute $m^\star = \lceil \frac{2}{\Delta^2}\log\frac{T\Delta^2}{2} \rceil$
(Tutorial 1, exercise 3) and store it in `M_STAR`. On a single figure, plot the mean
pseudo-regret, with bands, of uniform exploration, greedy, $\varepsilon$-greedy with
$\varepsilon = 0.1$ and ETC($m^\star$), using `regret_curves`. Print for each strategy the mean final regret
$\pm 2$ standard errors, and compare the one of ETC with the bound
$\frac{2}{\Delta}\big(1 + \log\frac{T\Delta^2}{2}\big)$.

In [ ]:
...  # TODO

**Question 8.** Plot the mean final regret (with $\pm 2$ standard error bars) of ETC as a
function of $m \in \{10, 25, 50, 100, 231, 500, 1000\}$, together with the bound
$g(m) = m\Delta + T\Delta e^{-m\Delta^2/2}$ of Tutorial 1. Do the same for
$\varepsilon$-greedy as a function of
$\varepsilon \in \{0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5\}$. Use a logarithmic x-axis.

In [ ]:
...  # TODO

## 6. UCB

At round $t > K$, the strategy UCB($\alpha$) plays the arm maximizing the index
$$U_a(t) = \hat\mu_a(t-1) + \sqrt{\frac{\alpha \log t}{N_a(t-1)}} \enspace,$$
and $\alpha = 2$ gives UCB1. During the first $K$ rounds, each arm is played once, in order.

**Question 9.** Complete `UCBAgent`. The method `index()` returns the array of the $K$
indices $U_a(t)$ of the next round, and is only called once every arm has been pulled. The
method `act()` plays arms $0, 1, \ldots, K-1$ during the first $K$ rounds, then the argmax
of the index, ties going to the smallest index.

*Hint:* when `act()` is called, `self.t` rounds have been played, so the next round is
round `self.t + 1`.

In [ ]:
class UCBAgent(Agent):
    def __init__(self, n_arms, alpha=2.0, rng=None):
        self.alpha = alpha
        super().__init__(n_arms, rng)

    def index(self):
        ...  # TODO

    def act(self):
        ...  # TODO


checks.check_ucb(UCBAgent)

**Question 10.** Run UCB1 once for $1000$ rounds on the reference instance, with the loop
below, and store at each round $t > K$ the indices $U_a(t)$ and the empirical means
$\hat\mu_a(t-1)$. Plot, as functions of $t$, the two indices (solid lines), the two
empirical means (dotted lines) and the true means (horizontal lines). Print the final
counts $N_a(1000)$.

In [ ]:
env = BernoulliBandit(MEANS, rng=0)
agent = UCBAgent(env.n_arms, rng=1)
n_rounds = 1000
indices = np.full((n_rounds, env.n_arms), np.nan)
means_hat = np.full((n_rounds, env.n_arms), np.nan)
for s in range(n_rounds):
    ...  # TODO

...  # TODO

## 7. Comparison with the naive strategies

**Question 11.** On a single figure, plot the mean pseudo-regret, with bands, of
$\varepsilon$-greedy with $\varepsilon = 0.1$, ETC with $m = m^\star$ (`M_STAR`) and
UCB1, using `regret_curves`. Print for each strategy the mean final regret $\pm 2$ standard
errors. Which of the three strategies use the value of $\Delta$ or of $T$?

In [ ]:
...  # TODO

**Question 12.** Run UCB1 with horizon $T = 20000$. Plot its mean pseudo-regret with a
logarithmic x-axis, together with the bound $8\log(t)/\Delta + \Delta$ of Tutorial 2
(exercise 2). Do the same for $\varepsilon$-greedy with $\varepsilon = 0.1$.

In [ ]:
...  # TODO

## 8. The exploration parameter

**Question 13.** Plot the mean final regret (with $\pm 2$ standard error bars) of
UCB($\alpha$) as a function of $\alpha \in \{0.1, 0.25, 0.5, 1, 2, 4, 8\}$, with a
logarithmic x-axis. For $\alpha = 0.1$ and $\alpha = 2$, plot the histogram of the final
regret over the runs.

In [ ]:
...  # TODO

## 9. Changing the gap

**Question 14.** Keep the tuning of the reference instance: $\varepsilon = 0.1$,
$m = 231$ and $\alpha = 2$. For $\Delta \in \{0.05, 0.1, 0.2, 0.4\}$, run the three
strategies on the instance with means $(0.6, 0.6 - \Delta)$ and plot the mean final regret
as a function of $\Delta$, with $\pm 2$ standard error bars.

In [ ]:
...  # TODO

## Going further

**Question 15.** Consider the instance with five Bernoulli arms of means
$(0.7, 0.6, 0.5, 0.4, 0.2)$. Run UCB1 for `N_RUNS` runs of horizon `HORIZON`, and compare,
for each suboptimal arm, the mean number of pulls $N_a(T)$ with the bound
$8\log(T)/\Delta_a^2 + 1$ of Tutorial 2. Which arm contributes most to the regret?

In [ ]:
...  # TODO

**Question 16.** In Question 13, a small $\alpha$ gives a small mean regret on the reference
instance. Run UCB($\alpha$) for $\alpha \in \{0.1, 0.5, 2\}$ on the instance with means
$(0.6, 0.55)$ and horizon $T = 20000$, and plot the mean pseudo-regret with the band
`band="quantile"` of `plot_mean`.

In [ ]:
...  # TODO

**Question 17.** Implement a variant of $\varepsilon$-greedy with
$\varepsilon_t = \min(1, c/t)$, and plot its final regret as a function of
$c \in \{1, 5, 20, 100, 500\}$. Does the best $c$ depend on the gap $\Delta$? Run the
experiment again with means $(0.6, 0.5)$.

In [ ]:
...  # TODO